In [ ]:
# !pip install google-genai -q

In [ ]:
import pandas as pd
import numpy as np

## Dataset (MBPP)

In [ ]:
sampled_dataset = '/kaggle/input/sanitized-mbpp-437/sanitized-mbpp-sample-100.json'

In [ ]:
# read jsonl files
df = pd.read_json(sampled_dataset, lines=True)
# df = df[20:25]

In [ ]:
df.shape[0]

## Constants

In [ ]:
MODEL_NAME = "gemini-2.5-flash"
GREEN_WORDS = ['billions', 'dlrs', 'shade', 'trade', 'profit']
RED_WORDS = ['market', 'year', 'company', 'revs']

BASE_DIR = '.'
VERSION = 2
SIZE = df.shape[0]
RESULTS_CSV = f'{BASE_DIR}/results/csv/exp_1_during_gen_v{VERSION}_{SIZE}.csv'
OUTPUT_DIR = f'{BASE_DIR}/output/output_{MODEL_NAME}_exp_1_dgen_v{VERSION}_{SIZE}'
Z_THRESHOLD = 4
GAMMA = 0.396158  # From NLTK Brown corpus

In [ ]:
green_letters = set([word[0] for word in GREEN_WORDS])
red_letters = set([word[0] for word in RED_WORDS])

print("GREEN LETTERS:", green_letters)
print("RED LETTERS:", red_letters)

## Helper Methods

In [ ]:
import ast
import os
import sys
import importlib.util
from collections import Counter
from pathlib import Path
import builtins

BUILTIN_NAMES = set(dir(builtins))

class CodeAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.public_classes = set()
        self.private_classes = set()
        self.public_functions = set()
        self.private_functions = set()
        self.public_variables = set()
        self.private_variables = set()

    def visit_FunctionDef(self, node):
        name = node.name
        if name.startswith("__") and name.endswith("__"):
            self.public_classes.add(name)
        elif name.startswith("_"):
            self.private_functions.add(name)
        else:
            self.public_functions.add(name)

        for arg in node.args.args:
            if arg.arg in {"self", "cls"}:
                self.public_variables.add(arg.arg)
            else:
                self.private_variables.add(arg.arg)
        self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name) and node.func.id in BUILTIN_NAMES:
            return
        self.generic_visit(node)

    def visit_ClassDef(self, node):
        if node.name.startswith("_"):
            self.private_classes.add(node.name)
        else:
            self.public_classes.add(node.name)
        self.generic_visit(node)

    def visit_Assign(self, node):
        for target in node.targets:
            if isinstance(target, ast.Name):
                if target.id.isupper():
                    self.public_variables.add(target.id)
                else:
                    self.private_variables.add(target.id)
        self.generic_visit(node)

    def visit_Attribute(self, node):
        if isinstance(node.value, ast.Name) and node.value.id in {"self", "cls"}:
            if node.attr not in BUILTIN_NAMES:
                if node.attr.startswith("_"):
                    self.private_variables.add(node.attr)
                else:
                    self.public_variables.add(node.attr)
        self.generic_visit(node)

def load_generated_code(file_path):
    if not os.path.exists(file_path):
        return None
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
    return content.strip() if content.strip() else None

def run_code_tests(code, test_cases):
    results = []
    temp_file = 'temp_test.py'
    
    with open(temp_file, 'w', encoding='utf-8') as f:
        f.write(code + '\n')
    
    try:
        spec = importlib.util.spec_from_file_location("temp_test", temp_file)
        module = importlib.util.module_from_spec(spec)
        sys.modules["temp_test"] = module
        spec.loader.exec_module(module)
        
        globals().update(vars(module))
        
        for idx, test in enumerate(test_cases, 1):
            try:
                exec(test, globals())
                results.append((idx, True, None))
            except Exception as e:
                results.append((idx, False, str(e)))
    
    except Exception as e:
        for idx in range(len(test_cases)):
            results.append((idx+1, False, f"Code error: {str(e)}"))
    
    finally:
        if os.path.exists(temp_file):
            os.remove(temp_file)
    
    return results

In [ ]:
def calculate_z_score(token_count, green_count, gamma=GAMMA):
    import math
    if token_count == 0 or gamma <= 0 or gamma >= 1:
        return float("nan")
    return (green_count - gamma * token_count) / math.sqrt(gamma * (1 - gamma) * token_count)

def extract_words_from_text(code_text):
    """Extract all words from code treating it as plain text"""
    import re
    # Extract all word-like tokens (alphanumeric sequences)
    words = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code_text)
    # Filter out very short words and common single characters
    filtered_words = [word for word in words if len(word) > 1]
    return filtered_words

def analyze_text_watermark(original_code, generated_code):
    """Analyze watermark using simple text-based approach"""
    
    def get_text_statistics(code):
        words = extract_words_from_text(code)
        if not words:
            return set(), 0, []
        
        # Get starting letters of all words
        starting_letters = [word[0].lower() for word in words if word and word[0].isalpha()]
        letter_set = set(starting_letters)
        
        # Count green letters (based on GREEN_WORDS first letters)
        green_letters = set([word[0].lower() for word in GREEN_WORDS])
        green_count = sum(1 for letter in starting_letters if letter in green_letters)
        
        z_score = calculate_z_score(len(starting_letters), green_count)
        
        print(f'Words found: {len(words)}')
        print(f'Starting letters: {len(starting_letters)}')
        print(f'Green letters count: {green_count}')
        print(f'Unique starting letters: {sorted(letter_set)}')
        
        return letter_set, z_score, words

    preferred = set([word[0].lower() for word in GREEN_WORDS])
    avoided = set([word[0].lower() for word in RED_WORDS])

    print("Preferred letters:", preferred)
    print("Avoided letters:", avoided)
    print("\n--- ORIGINAL CODE ---")
    orig_letters, orig_z, orig_words = get_text_statistics(original_code)
    print("\n--- GENERATED CODE ---")
    gen_letters, gen_z, gen_words = get_text_statistics(generated_code)

    orig_pref_ratio = len(orig_letters & preferred) / len(orig_letters) if orig_letters else 0
    gen_pref_ratio = len(gen_letters & preferred) / len(gen_letters) if gen_letters else 0
    
    orig_avoid_count = len(orig_letters & avoided)
    gen_avoid_count = len(gen_letters & avoided)
    
    return {
        "original_preferred_ratio": orig_pref_ratio,
        "generated_preferred_ratio": gen_pref_ratio,
        "original_z_score": orig_z,
        "generated_z_score": gen_z,
        "original_is_watermarked": orig_z >= Z_THRESHOLD,
        "generated_is_watermarked": gen_z >= Z_THRESHOLD,
        "original_avoided_count": orig_avoid_count,
        "generated_avoided_count": gen_avoid_count,
        "original_letters": sorted(orig_letters),
        "generated_letters": sorted(gen_letters),
        "original_word_count": len(orig_words),
        "generated_word_count": len(gen_words)
    }

## Watermark Embedding

### Gemini-2.5-Flash

In [ ]:
from google import genai
from google.genai import types
import os
import re
from pathlib import Path
import ast


API_KEY = os.getenv("GEMINI_API_KEY") or os.getenv("API_KEY")

if not API_KEY:
    raise RuntimeError(
        "Missing API key. Please set GEMINI_API_KEY or API_KEY in your environment (e.g., .env)."
    )

client = genai.Client(api_key=API_KEY)

## PROMPT TEMPLATE REFERENCE:

```tex
Wang, C. Y., DaghighFarsoodeh, A., & Pham, H. V. (2024). 
Selection of prompt engineering techniques for code generation through predicting code complexity. 
arXiv preprint arXiv:2409.16416.```

In [ ]:
SYSTEM_PROMPT = '''
### Green Letter List: {green_words}
### Red Letter List: {red_words}
### Command:
Generate responses that are:
1. Clear & Coherent: Easy to follow and logically organized.
2. Accurate & Concrete: Provides precise facts, examples, or steps.
3. Contextually Relevant: Directly addresses the user's intent and context.
4. "Green Letter" Enriched: Increase use of words beginning with Green List letters, reduce Red List letters.
'''

PROBLEM_TEMPLATE = (
    "You are a helpful code assistant that can teach a junior developer how to code. Your language of choice is Python. Only generate the Python code for the following task.\n\n"
    "##Problem:\n{prompt}\n\n"
    "##Test Cases:\n{tests}\n\n"
)

def generate_code(record):
    prompt = record["prompt"]
    tests = "\n".join(record["test_list"]) 
    system_instruction = SYSTEM_PROMPT.format(green_words=GREEN_WORDS, red_words=RED_WORDS)
    full_prompt = PROBLEM_TEMPLATE.format(prompt=prompt, tests=tests)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=full_prompt)],
        )
    ]

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=contents,
        config=types.GenerateContentConfig(system_instruction=system_instruction),
    )

    text = response.text
    
    # Find all python code blocks
    code_blocks = re.findall(r'```python(.*?)```', text, re.DOTALL)

    # Get the last (actual code) block, or the one with actual content
    actual_code_blocks = [block for block in code_blocks if block.strip()]
    if actual_code_blocks:
        text = actual_code_blocks[-1].strip()  # Get the last non-empty block
    else:
        print("No actual code blocks found")
        text = ''

    print('Generated:', len(text))
    print('=' * 50)
    return (text or "").strip()

### Execution

In [ ]:
def process_dataset(df, output_dir):
    Path(output_dir).parent.mkdir(exist_ok=True)
    results = []
    
    for _, record in df.iterrows():
        task_id = record.get('task_id') or record.get('id') or str(_)
        output_file = Path(output_dir) / f"{task_id}.py"
        original_code = record.get('code', '')

        try:
            generated_code = generate_code(record)
            output_file.parent.mkdir(parents=True, exist_ok=True)  # Ensure directory exists
            
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(generated_code)
        except Exception as e:
            print(f"Generation failed for {task_id}: {e}")
            if not output_file.exists():
                results.append({
                    'task_id': task_id, 
                    'status': 'failed',
                    'tests_passed': 0, 'tests_failed': 0, 'total_tests': 0,
                    'pass_rate': 0, 'all_passed': False
                })
                continue

        if output_file.exists():
            generated_code = output_file.read_text(encoding='utf-8')
            
            watermark_analysis = analyze_text_watermark(original_code, generated_code)
            
            test_results = run_code_tests(generated_code, record['test_list'])
            passed = sum(1 for _, success, _ in test_results if success)
            total = len(test_results)
            failed = total - passed
            pass_rate = (passed / total * 100) if total > 0 else 0
            all_passed = (passed == total)
            
            result = {
                'task_id': task_id,
                'tests_passed': passed, 'tests_failed': failed, 'total_tests': total,
                'pass_rate': round(pass_rate, 2), 'all_passed': all_passed,
                **watermark_analysis
            }
            results.append(result)
        else:
            results.append({
                'task_id': task_id, 'status': 'missing',
                'tests_passed': 0, 'tests_failed': 0, 'total_tests': 0,
                'pass_rate': 0, 'all_passed': False
            })

    results_df = pd.DataFrame(results)
    Path(RESULTS_CSV).parent.mkdir(parents=True, exist_ok=True)    
    results_df.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved to {RESULTS_CSV}")
    return results

In [ ]:
results = process_dataset(df, OUTPUT_DIR)

## Watermark Detection

- nltk:brown: total_words=1,013,640  gamma=**0.396158**
- nltk:gutenberg: total_words=2,136,080  gamma=**0.399109**
- nltk:webtext: total_words=306,832  gamma=**0.323187**

### Calculate Final Results

In [ ]:
analysis_results = []

for _, row in df.iterrows():
    task_id = row['task_id']
    original_code = row['code']
    
    print(f'Task: {task_id}')
    
    generated_file = Path(OUTPUT_DIR) / f"{task_id}.py"
    if not generated_file.exists():
        print(f"Missing file for {task_id}")
        continue

    generated_code = generated_file.read_text(encoding='utf-8')
    watermark_result = analyze_text_watermark(original_code, generated_code)
    
    test_results = run_code_tests(generated_code, row['test_list'])
    passed = sum(1 for _, success, _ in test_results if success)
    total = len(test_results)
    
    result = {
        'task_id': task_id,
        'tests_passed': passed, 'tests_failed': total - passed, 'total_tests': total,
        'pass_rate': round((passed / total * 100) if total > 0 else 0, 2),
        'all_passed': (passed == total),
        **watermark_result
    }
    analysis_results.append(result)

In [ ]:
results_df = pd.DataFrame(analysis_results)
results_df.to_csv(RESULTS_CSV, index=False)
print(f"Saved: {RESULTS_CSV}")

print(f"\nCode Quality Summary:")
print(f"Total: {len(results_df)}")
print(f"Avg pass rate: {results_df['pass_rate'].mean():.1f}%")
print(f"All tests passed: {results_df['all_passed'].sum()}")
print(f"Some failures: {(~results_df['all_passed']).sum()}")